In [1]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [2]:
g=Symp_symb(7)
C=g.cochain_complex
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis
P=RegularCartanGeometry(g,'eta')
D=Distr_of_constant_symbol(g,-P.curvature)
eta=IndexedBase('eta')
P.fund_invars=[eta[3,9,6],eta[6,9,9],eta[3,9,4]]

In [3]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

### Computations

In [4]:
Bianchi_dict={}
not_added=[]
rel_Bianchi_dict={}
not_added_rel_Bianchi_dict={}

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve,rel_keys=ds_subs(zero_elt.vec[m],Bianchi_dict,D)
        rel_Bianchi_keys=set()
        for a in rel_keys: rel_Bianchi_keys=rel_Bianchi_keys.union(rel_Bianchi_dict[a[0]][a[1]])
        rel_Bianchi_keys.add((i,j,k,m))

        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,P.fund_invars)
            if s==None: s=find_a_linear_term(to_solve)
            if s==None:
                print('no linear term in',t)
                not_added.append(t)
                rel_Bianchi_dict[t]=rel_Bianchi_keys
            else:
                sol=solve(to_solve,s,dict=True)[0]
                print('    Solving complete in',hrs_min_sec(time.time()-time3))
                time4=time.time()
                for a in sol: 
                    ds_add_key(a,sol[a],Bianchi_dict,D,rel_Bianchi_keys,rel_Bianchi_dict,not_added)
                print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D.curv = -P.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D)[0])
        if r!=0: print(r)

In [ ]:
for w in range(1,10):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    # In order to track what identities are being used, I won't want to substitute here. It takes longer though :/
    # P.update_fund_ders(Bianchi_dict,D)
    # P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    # D.curv=-P.curvature[0]
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
# not_added_bookmark=copy.copy(not_added)
# P_curv_bookmark=copy.deepcopy(P.curvature)
# Bianchi_dict_bookmark=copy.deepcopy(Bianchi_dict)
# rel_Bianchi_dict_bookmark=copy.deepcopy(rel_Bianchi_dict)

In [5]:
# # Shelve
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Bianchi_dict_bookmark'+'{j}'.format(j=7)]=copy.deepcopy(Bianchi_dict)
#     shelf['P_curv_bookmark'+'{j}'.format(j=7)]=copy.deepcopy(P.curvature)
#     shelf['not_added_bookmark'+'{j}'.format(j=7)]=copy.copy(not_added)
#     shelf['rel_Bianchi_dict_bookmark'+'{j}'.format(j=7)]=copy.copy(rel_Bianchi_dict)

# Unshelve
with shelve.open('Abstract_Syzygies') as shelf:
    Bianchi_dict=shelf['Bianchi_dict_bookmark'+'{j}'.format(j=7)]
#     P.curvature=shelf['P_curv_bookmark'+'{j}'.format(j=7)]
    not_added=shelf['not_added_bookmark'+'{j}'.format(j=7)]
    rel_Bianchi_dict=copy.deepcopy(shelf['rel_Bianchi_dict_bookmark'+'{j}'.format(j=7)])

In [7]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

False
[]


#### Temporary

In [6]:
B_L={}
for a in rel_Bianchi_dict[eta[5,8,9]][tuple([])]:
    B_L[a]=Bianchi(P,*(a[0:3])).vec[a[3]]

KeyboardInterrupt: 

In [44]:
Bianchi_dict[eta[5,8,9]][tuple([])]

0

In [38]:
def check_minimal_generators(expr,gen_list):
    for i in range(len(gen_list)):
        print(gen_list[i])
        temp_ds_dict={}
        temp_gl=gen_list[0:i]+gen_list[i+1:len(gen_list)]
        for t in temp_gl:
            print('    temp_ds_dict:',temp_ds_dict,'\n')
            new_expr=ds_subs(Bianchi(P,*t[0:3]).vec[t[3]],temp_ds_dict,D)[0]
            print('    Adding',t,':',new_expr,'\n\n')
            add_expr_to_ds_dict(new_expr,temp_ds_dict,D)
        test=ds_subs(expr,temp_ds_dict,D)[0]
        print('    test =',test)
        if test==0:
            return gen_list[i]
    return []

In [43]:
check_minimal_generators(eta[5,8,9],list(B_L.keys()))

(3, 4, 5, 6)
    temp_ds_dict: {} 

    Adding (4, 5, 8, 10) : eta[4, 6, 6] - 2*eta[4, 8, 8] + 9*eta[5, 6, 7]/5 + 8*eta[5, 7, 8]/5 - 2*eta[5, 8, 9] + 2*eta[6, 7, 9] 


    temp_ds_dict: {eta[4, 8, 8]: {(): eta[4, 6, 6]/2 + 9*eta[5, 6, 7]/10 + 4*eta[5, 7, 8]/5 - eta[5, 8, 9] + eta[6, 7, 9]}} 

    Adding (3, 4, 7, 8) : -2*eta[4, 6, 6] - 3*eta[4, 9, 9] - 18*eta[5, 6, 7]/5 - 21*eta[5, 7, 8]/5 + 3*eta[5, 8, 9] - 4*eta[6, 7, 9] 


    temp_ds_dict: {eta[4, 8, 8]: {(): -3*eta[4, 9, 9]/4 - eta[5, 7, 8]/4 - eta[5, 8, 9]/4}, eta[6, 7, 9]: {(): -eta[4, 6, 6]/2 - 3*eta[4, 9, 9]/4 - 9*eta[5, 6, 7]/10 - 21*eta[5, 7, 8]/20 + 3*eta[5, 8, 9]/4}} 

    Adding (3, 5, 7, 9) : eta[4, 6, 6]/2 + 3*eta[4, 9, 9]/4 + 9*eta[5, 6, 7]/10 + 41*eta[5, 7, 8]/20 - 7*eta[5, 8, 9]/4 


    temp_ds_dict: {eta[4, 8, 8]: {(): -3*eta[4, 9, 9]/4 - eta[5, 7, 8]/4 - eta[5, 8, 9]/4}, eta[6, 7, 9]: {(): eta[5, 7, 8] - eta[5, 8, 9]}, eta[4, 6, 6]: {(): -3*eta[4, 9, 9]/2 - 9*eta[5, 6, 7]/5 - 41*eta[5, 7, 8]/10 + 7*eta[5, 8, 9]/2}

[]

In [51]:
# This really is a minimal set of generators :(
for a in B_L:
    print(a)
    print(B_L[a])
    print()

(3, 4, 5, 6)
-3*eta[4, 6, 6] + 2*eta[4, 8, 8] + 2*eta[4, 9, 9] - 27*eta[5, 6, 7]/5 - 24*eta[5, 7, 8]/5 - eta[5, 8, 9] - 2*eta[6, 7, 9]

(4, 5, 8, 10)
eta[4, 6, 6] - 2*eta[4, 8, 8] + 9*eta[5, 6, 7]/5 + 8*eta[5, 7, 8]/5 - 2*eta[5, 8, 9] + 2*eta[6, 7, 9]

(3, 4, 7, 8)
-eta[4, 6, 6] - 2*eta[4, 8, 8] - 3*eta[4, 9, 9] - 9*eta[5, 6, 7]/5 - 13*eta[5, 7, 8]/5 + eta[5, 8, 9] - 2*eta[6, 7, 9]

(3, 5, 7, 9)
eta[5, 7, 8] - eta[5, 8, 9] - eta[6, 7, 9]

(3, 4, 6, 7)
3*eta[4, 8, 8] + 5*eta[4, 9, 9] - 32*eta[5, 6, 7]/5 - 24*eta[5, 7, 8]/5 - 3*eta[5, 8, 9]

(3, 5, 6, 8)
eta[5, 6, 7] - eta[5, 7, 8]

(3, 4, 8, 9)
-eta[4, 6, 6] + 2*eta[4, 8, 8] - 18*eta[5, 6, 7]/5 - 16*eta[5, 7, 8]/5 - 2*eta[5, 8, 9] - eta[6, 7, 9]



In [ ]:
for a in rel_Bianchi_dict:
    for b in rel_Bianchi_dict[a]:
        print(a,b,'-->',rel_Bianchi_dict[a][b],'\n')

In [ ]:
eta[6,9,9,4,4]-Bianchi_dict[eta[6,9,9]][(4,4)]

In [ ]:
for a in rel_Bianchi_dict[eta[6,9,9]][(4,4)]:
    temp=ds_subs(P.Bianchi_cache[a[0:3]].vec[a[3]],Bianchi_dict,D)[1]
    if len(temp)<6:
        u=set().union(*[rel_Bianchi_dict[b[0]][b[1]] for b in temp])
        print(a,'-->',u)
        print(P.Bianchi_cache[a[0:3]].vec[a[3]],'=',ds_subs(P.Bianchi_cache[a[0:3]].vec[a[3]],Bianchi_dict,D)[0],'\n')

In [ ]:
print(rel_Bianchi_dict[eta[3,9,6]][(4,)])
Bianchi_dict[eta[3,9,6]][(4,)] 

##### Which components have leading term $I^3$?

In [83]:
candidates=set()
for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        for term in Bianchi_dict[k][j].as_coeff_add()[1]:
            if term.as_coeff_Mul()[1]==eta[6,9,9]**3:
                candidates=candidates.union(rel_Bianchi_dict[k][j])

In [99]:
Bianchi_dict_8=copy.deepcopy(Bianchi_dict)
for k in list(Bianchi_dict_8.keys()):
    for j in list(Bianchi_dict_8[k].keys()):
        if -wght_of_ind(k.base[k.indices+j],g)>8: del Bianchi_dict_8[k][j]
    if Bianchi_dict_8[k]=={}: del Bianchi_dict_8[k]

In [105]:
I3LT=set()
for a in candidates:
    expr=expand(ds_subs(Bianchi(P,*a[0:3]).vec[a[3]],Bianchi_dict_8,D)[0])
    print(expr)
    for term in expr.as_coeff_add()[1]:
        if term.as_coeff_Mul()[1]==eta[6,9,9]**3:
            I3LT.add(a)
    if a in I3LT: print(a,'added')
    else: print(a,'not added')

0
(3, 5, 7, 0) not added
0
(3, 5, 7, 9) not added
0
(3, 5, 10, 5) not added
0
(3, 6, 10, 10) not added
0
(3, 7, 9, 9) not added
0
(3, 4, 5, 6) not added
0
(3, 4, 8, 2) not added
0
(3, 6, 7, 4) not added
0
(3, 5, 7, 2) not added
0
(3, 5, 10, 7) not added
0
(3, 4, 8, 4) not added
0
(3, 5, 7, 4) not added
0
(3, 5, 10, 9) not added
0
(3, 7, 8, 6) not added
0
(3, 5, 6, 5) not added
-41*eta[3, 9, 4, 4, 4, 4]/2664 - 392722*eta[3, 9, 6, 3, 3, 4, 4, 4]/5303025 - 1056714419*eta[3, 9, 6, 3, 4, 4, 3, 4]/10075747500 - 197*eta[3, 9, 6, 3, 4, 4, 4, 3]/3080 - eta[4, 10, 0] - 967983979*eta[6, 9, 9, 3, 3, 3, 3, 4, 4]/439393500 + 386782481*eta[6, 9, 9, 3, 3, 3, 4, 3, 4]/564086250 - 100657*eta[6, 9, 9, 3, 3, 3, 4, 4, 3]/143550 - 138773*eta[6, 9, 9, 3, 3, 4]*eta[6, 9, 9]/70300 - 653462168147*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 4]/275499724500 + 7*eta[6, 9, 9, 3, 4, 3, 3, 3, 4]/15 + 43*eta[6, 9, 9, 3, 4, 3]*eta[6, 9, 9]/15 - 433*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 3]/57 + 12616*eta[6, 9, 9, 3]*eta[6, 9, 9, 4, 3]/24

In [120]:
I3LT=list(I3LT)
I3LT

[(4, 6, 7, 0), (4, 5, 9, 1), (4, 5, 8, 0)]

In [121]:
A1,A2,A3=[expand(ds_subs(Bianchi(P,*a[0:3]).vec[a[3]],Bianchi_dict_8,D)[0]) for a in I3LT]

In [122]:
(A1+A3)*Rational(12740,309)

-eta[3, 9, 6, 3, 3, 4, 4, 4] + 3*eta[3, 9, 6, 3, 4, 4, 3, 4] - 2*eta[3, 9, 6, 3, 4, 4, 4, 3] - 7438508*eta[6, 9, 9, 3, 3, 3, 3, 4, 4]/403245 + 14877016*eta[6, 9, 9, 3, 3, 3, 4, 3, 4]/403245 - 7438508*eta[6, 9, 9, 3, 3, 3, 4, 4, 3]/403245 - 60572848*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 4]/2016225

In [125]:
A2*Rational(25,36)

25*eta[3, 9, 4, 4, 4, 4]/648 + 48275*eta[3, 9, 6, 3, 3, 4, 4, 4]/825552 + 85829333*eta[3, 9, 6, 3, 4, 4, 3, 4]/568598940 + 9770880109*eta[3, 9, 6, 3, 4, 4, 4, 3]/200146826880 + 25*eta[5, 10, 1]/36 + 6664975*eta[6, 9, 9, 3, 3, 3, 3, 4, 4]/3420144 - 1497449*eta[6, 9, 9, 3, 3, 3, 4, 3, 4]/6248340 + 90361147*eta[6, 9, 9, 3, 3, 3, 4, 4, 3]/148918770 + 5255*eta[6, 9, 9, 3, 3, 4]*eta[6, 9, 9]/1368 - 28454009*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 4]/112864752 - 25*eta[6, 9, 9, 3, 4, 3, 3, 3, 4]/108 - 20743*eta[6, 9, 9, 3, 4, 3]*eta[6, 9, 9]/4104 + 50117*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 3]/4104 - 1095755*eta[6, 9, 9, 3]*eta[6, 9, 9, 4, 3]/135432 - 40*eta[6, 9, 9, 4, 3, 3, 3, 3, 4]/81 + 89*eta[6, 9, 9, 4, 3, 3]*eta[6, 9, 9]/3564 - eta[6, 9, 9]**3

In [124]:
A3

41*eta[3, 9, 4, 4, 4, 4]/2664 + 1056403*eta[3, 9, 6, 3, 3, 4, 4, 4]/21212100 + 447463886*eta[3, 9, 6, 3, 4, 4, 3, 4]/2518936875 + 4331*eta[3, 9, 6, 3, 4, 4, 4, 3]/280280 + eta[4, 10, 0] + 257131613*eta[6, 9, 9, 3, 3, 3, 3, 4, 4]/146464500 + 39324473*eta[6, 9, 9, 3, 3, 3, 4, 3, 4]/188028750 + 663049*eta[6, 9, 9, 3, 3, 3, 4, 4, 3]/2612610 + 138773*eta[6, 9, 9, 3, 3, 4]*eta[6, 9, 9]/70300 + 452715096611*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 4]/275499724500 - 7*eta[6, 9, 9, 3, 4, 3, 3, 3, 4]/15 - 43*eta[6, 9, 9, 3, 4, 3]*eta[6, 9, 9]/15 + 433*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 3]/57 - 12616*eta[6, 9, 9, 3]*eta[6, 9, 9, 4, 3]/2475 - 7*eta[6, 9, 9, 4, 3, 3, 3, 3, 4]/45 + 1403*eta[6, 9, 9, 4, 3, 3]*eta[6, 9, 9]/2475 - 84*eta[6, 9, 9]**3/125

In [71]:
syzygies_by_wght[9]

[-5*eta[3, 9, 6, 3, 3, 4, 4, 4] + 15*eta[3, 9, 6, 3, 4, 4, 3, 4] - 10*eta[3, 9, 6, 3, 4, 4, 4, 3] - 224*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 4],
 -5*eta[3, 9, 6, 3, 3, 3, 4, 4] + 10*eta[3, 9, 6, 3, 3, 4, 3, 4] - 5*eta[3, 9, 6, 3, 3, 4, 4, 3] + 24*eta[3, 9, 6, 3]*eta[6, 9, 9, 4],
 -35*eta[3, 9, 4, 3, 4, 4] + 210*eta[3, 9, 4, 4, 4, 3] - 440*eta[3, 9, 6, 3, 3, 3, 4, 4] - 825*eta[3, 9, 6, 3, 3, 4, 4, 3] + 2475*eta[3, 9, 6, 3, 4, 3, 3, 4] - 792*eta[3, 9, 6, 3, 4]*eta[6, 9, 9] + 3168*eta[3, 9, 6, 3]*eta[6, 9, 9, 4] - 6336*eta[3, 9, 6]*eta[6, 9, 9, 3, 4] + 9240*eta[6, 9, 9, 3, 3, 3, 3, 3, 4] + 44352*eta[6, 9, 9, 3, 3]*eta[6, 9, 9, 3],
 -280*eta[3, 9, 4, 3, 3, 4] + 455*eta[3, 9, 4, 3, 4, 3] - 784*eta[3, 9, 4]*eta[6, 9, 9] - 280*eta[3, 9, 6, 3, 3, 3, 3, 4] + 1925*eta[3, 9, 6, 3, 3, 3, 4, 3] - 4950*eta[3, 9, 6, 3, 3, 4, 3, 3] - 1144*eta[3, 9, 6, 3, 3]*eta[6, 9, 9] + 5775*eta[3, 9, 6, 3, 4, 3, 3, 3] - 10296*eta[3, 9, 6, 3]*eta[6, 9, 9, 3] - 6912*eta[3, 9, 6]*eta[6, 9, 9, 3, 3] + 18480*eta[6, 9, 9, 3, 

#### End Temporary

### General Case

#### Unbranched 

In [8]:
syzygies_by_wght={}
for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if not w in syzygies_by_wght: syzygies_by_wght[w]=[]
        new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
        if new_syzygy!=0:
            new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
            syzygies_by_wght[w].append(new_syzygy)

In [9]:
temp_dict={eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}}
for a in syzygies_by_wght[5]:
        expr=ds_subs(a,temp_dict,D)[0]
        if expr!=0:
            display(expr)

7*eta[6, 9, 9, 3, 3]

eta[6, 9, 9, 4, 4]

In [10]:
ess_syz={eta[3,9,6,4],eta[6,9,9,4,4]}
for a in ess_syz:
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    display(ds_subs(a-Bianchi_dict[k][j],temp_dict,D)[0])

eta[6, 9, 9, 4, 4]

7*eta[6, 9, 9, 3, 3]

In [11]:
# # What is reprocessed when I add Wilc=0 and the first essential syzygies?
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},eta[6,9,9]: {(3,3): 0,(4,4):0}}

for k in Bianchi_dict:
    for j in Bianchi_dict[k]:
        rel=-k.base[k.indices+j]+Bianchi_dict[k][j]
        new_rel=ds_subs(rel,temp_dict,D)[0]
        if k in temp_dict and new_rel!=0:
            for i in temp_dict[k]:
                if j[0:len(i)]==i:
                    display(k.base[k.indices+j])
                    display(new_rel)
                    display('--------------------------------')

eta[6, 9, 9, 3, 3, 4, 4]

2*eta[6, 9, 9, 3, 4, 3, 4] - 4*eta[6, 9, 9, 4]*eta[6, 9, 9]/15

'--------------------------------'

eta[6, 9, 9, 3, 3, 3, 4, 4, 4]

424216*eta[6, 9, 9, 3, 4, 3, 4, 4, 3]/860003 + 3749004*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 4]/4300015 - 3571366*eta[6, 9, 9, 4, 3]*eta[6, 9, 9, 4]/12900045

'--------------------------------'

In [12]:
ess_syz.add(eta[6,9,9,3,3,4,4])
for a in ess_syz:
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    display(ds_subs(a-Bianchi_dict[k][j],temp_dict,D)[0])

0

0

-2*eta[6, 9, 9, 3, 4, 3, 4] + 4*eta[6, 9, 9, 4]*eta[6, 9, 9]/15

In [13]:
# Syzygies
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,4):0,(3,4,3,4):Rational(2,15)*eta[6,9,9,4]*eta[6,9,9]}}

for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if w in [5,6,7,8,9]:
            new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
            if ds_subs(new_syzygy,temp_dict,D)[0]!=0:
                new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
                display(k.base[k.indices+j])
                display(ds_subs(new_syzygy,temp_dict,D)[0])
                display('------------------------------------------')
            

eta[6, 9, 9, 3, 4, 4]

eta[6, 9, 9, 3, 4, 4]

'------------------------------------------'

eta[6, 9, 9, 4, 3, 4]

eta[6, 9, 9, 4, 3, 4]

...

-629832672*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 4] + 104972112*eta[6, 9, 9, 4, 3]*eta[6, 9, 9, 4]

'------------------------------------------'

eta[6, 9, 9, 3, 4, 3, 3, 3, 4]

92400*eta[6, 9, 9, 3, 4, 3, 3, 3, 4] + 184800*eta[6, 9, 9, 3, 4, 3]*eta[6, 9, 9] + 404880*eta[6, 9, 9, 3]*eta[6, 9, 9, 4, 3] - 77840*eta[6, 9, 9, 4, 3, 3]*eta[6, 9, 9] + 22176*eta[6, 9, 9]**3

'------------------------------------------'

In [14]:
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,4):0,(3,4,3,4):Rational(2,15)*eta[6,9,9,4]*eta[6,9,9]}}
ess_syz.add(eta[6,9,9,3,3,3,4,4,4])
for a in ess_syz:
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    display(a)
    display(factor(ds_subs(a-Bianchi_dict[k][j],temp_dict,D)[0]))

eta[6, 9, 9, 4, 4]

0

eta[6, 9, 9, 3, 3, 3, 4, 4, 4]

-624834*(6*eta[6, 9, 9, 3, 4] - eta[6, 9, 9, 4, 3])*eta[6, 9, 9, 4]/4300015

eta[3, 9, 6, 4]

0

eta[6, 9, 9, 3, 3, 4, 4]

0

#### First Branch
Assuming $\eta_{69;4}^9=0$

In [17]:
not_added=copy.copy(not_added_bookmark)
P.curvature=copy.deepcopy(P_curv_bookmark)
Bianchi_dict=copy.deepcopy(Bianchi_dict_bookmark)
rel_Bianchi_dict=copy.deepcopy(rel_Bianchi_dict_bookmark)

In [18]:
not_added=copy.copy(not_added_bookmark)
ds_add_key(eta[6,9,9,4],0,Bianchi_dict,D,rel_Bianchi_keys={'branch 1'})

TypeError: argument of type 'NoneType' is not iterable

In [467]:
ess_syz_b1=set()
# Let's seek out the syzygies again using the assumption from this branch

In [ ]:
# Syzygies
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,):0,(3,4,3,4):0}}

for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if w in [5,6,7,8,9]:
            new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
            if ds_subs(new_syzygy,temp_dict,D)!=0:
                new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
                display(k.base[k.indices+j])
                display(ds_subs(new_syzygy,temp_dict,D)[0])
                display('------------------------------------------')

In [ ]:
ess_syz

In [ ]:
ess_syz_b1.add(eta[6,9,9,4,3,3,3,4])
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,4):0,(3,4,3,4):Rational(2,15)*eta[6,9,9,4]*eta[6,9,9]}}

for a in ess_syz.union(ess_syz_b1):
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    if k in Bianchi_dict and j in Bianchi_dict[k]:
        print(a)
        display(ds_subs(a-Bianchi_dict[k][j],temp_dict,D)[0])

In [ ]:
# # Reprocessed keys for eta[6,9,9,3,4]=0
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(3,4):0,(4,):0}}

for k in Bianchi_dict:
    for j in Bianchi_dict[k]:
        rel=-k.base[k.indices+j]+Bianchi_dict[k][j]
        new_rel=ds_subs(rel,temp_dict,D)
        if k in temp_dict and new_rel!=0:
            for i in temp_dict[k]:
                if j[0:len(i)]==i:
                    display(k.base[k.indices+j])
                    display(new_rel)
                    display('--------------------------------')        

In [ ]:
ess_syz_b1.add(eta[6,9,9,4,3,3,3,3,4])
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(3,4):0,(4,):0}}

for a in ess_syz.union(ess_syz_b1):
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    display(ds_subs(a-Bianchi_dict[k][j],temp_dict,D))

In [ ]:
for a in ess_syz.union(ess_syz_b1):
    k=a.base[a.indices[0:3]]
    j=a.indices[3:len(a.indices)]
    display(ds_subs(a-Bianchi_dict[k][j],temp_dict,D))

#### Second Branch
Assuming $\eta_{69;43}^9=6\eta_{69;34}^9$

In [60]:
ess_syz_b2=set()

In [ ]:
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,4):0,(4,3): 6*eta[6,9,9,3,4],(3,4,3,4):Rational(2,15)*eta[6,9,9,4]*eta[6,9,9]}}
ess_syz_b2.add(eta[6,9,9,4,3,3,4]) # This puts us back in the branch 1 case
display(ds_subs(eta[6,9,9,4,3,3,4]-Bianchi_dict[eta[6,9,9]][(4,3,3,4)],temp_dict,D))
eta[6,9,9,4,3,3,4] in ess_syz_b1 # This is really a necessary syzygy

### Finding the general syzygy

First use ess_syz to show $\eta_{69;33}^9$ is a function of Wilc and find the branching relation

Second, use ess_syz_b2 to show $\eta_{69;4}^9$ is a function of Wilc.

Third, use all essential syzygies to show $\eta_{69;34}^9$ is a function of Wilc.

Finally, show $\eta_{69}^9$ is a function of Wilc.

In [387]:
final_subs_dict={eta[6,9,9]:{}}
ess_syz_dict={}
for a in ess_syz.union(ess_syz_b1).union(ess_syz_b2):
    ess_syz_dict[a]=a-Bianchi_dict[a.base[a.indices[0:3]]][a.indices[3:len(a.indices)]]

In [395]:
final_subs_dict[eta[6,9,9]][(3,3)]=solve(ess_syz_dict[eta[3,9,6,4]],eta[6,9,9,3,3])[0]
final_subs_dict[eta[6,9,9]][(4,4)]=solve(ess_syz_dict[eta[6,9,9,4,4]],eta[6,9,9,4,4])[0]
final_subs_dict[eta[6,9,9]][(3,4,3,4)]=solve(ess_syz_dict[eta[6,9,9,3,3,4,4]],eta[6,9,9,3,4,3,4])[0]
final_subs_dict[eta[6,9,9]][(4,3,3,4)]=solve(ess_syz_dict[eta[6,9,9,4,3,3,4]],eta[6,9,9,4,3,3,4])[0]
final_subs_dict[eta[6,9,9]][(4,3,3,3,4)]=solve(ess_syz_dict[eta[6,9,9,4,3,3,3,4]],eta[6,9,9,4,3,3,3,4])[0]

In [389]:
zw_dict={eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}}

In [ ]:
a1=factor(ds_subs(ds_subs(ess_syz_dict[eta[6,9,9,3,3,3,4,4,4]],final_subs_dict,D),zw_dict,D)*Rational(4300015,624834))
a2=ds_subs(ds_subs(ess_syz_dict[eta[6,9,9,4,3,3,3,3,4]],final_subs_dict,D),zw_dict,D)*55-Rational(396,5)*eta[6,9,9]**3

# a1=factor(ds_subs(ess_syz_dict[eta[6,9,9,3,3,3,4,4,4]],final_subs_dict,D)*Rational(4300015,624834))
# a2=ds_subs(ess_syz_dict[eta[6,9,9,4,3,3,3,3,4]],final_subs_dict,D)*55*5

In [ ]:
display(a1)
display(factor(a2))

In [ ]:
for a in L:
    display(a)

In [ ]:
L=expand(a2**2).as_coeff_add()[1]
for i in range(len(L)):
    print(i,'-->',L[i])

In [ ]:
49500*6

In [ ]:
# To do: Factor into a product from the two ideals!
f=Rational(-1,6)*eta[6,9,9,3,4]/eta[6,9,9,4,3]
rel1=6*eta[6,9,9,3,4]-eta[6,9,9,4,3]
rel2=eta[6,9,9,4]

r=expand(a2**2)
# display(r)
display(collect(r,eta[6, 9, 9, 4, 3, 3, 3, 3, 4]))

# t1=factor(L[7]+f*L[7])*Rational(-3*39072000,6845000)
# print()
# print(t1)
# r+=t1
# r=expand(r)

L=r.as_coeff_add()[1]
for i in range(len(L)):
    print(i,'-->',L[i])

In [ ]:
for L in [[3],[4],[3,3],[3,4],[4,3],[4,4]]:
    expr=copy.copy(a1)
    for i in L:
        expr=P.fund_der(expr,i)
    display(L)
    display(expr)
    display('--------------------------')

In [ ]:
for a in ess_syz_dict:
    expr=ds_subs(ess_syz_dict[a],final_subs_dict,D)
    if expr!=0:
        display(a)
        display(ds_subs(expr,zw_dict,D))
        display('------------------')

In [ ]:
print(temp_dict)

In [ ]:
print(final_subs_dict)

In [ ]:
temp_dict=copy.deepcopy(final_subs_dict)
temp_dict[eta[6,9,9]].pop((4,4),None)
temp_dict[eta[6,9,9]].pop((4,3,3,4),None)
temp_dict[eta[6,9,9]].pop((4,3,3,3,4),None)
temp_dict[eta[6,9,9]][(4,)]=0

for a in ess_syz_dict:
    expr=ds_subs(ess_syz_dict[a],temp_dict,D)
    if expr!=0:
        display(a)
        display(expr)
        display('------------------')

In [ ]:
# Find the branching relation

rel1=ds_subs(P.fund_der(P.fund_der(ess_syz_dict[eta[6,9,9,3,3,4,4]],4),3),final_subs_dict,D)
rel2=ds_subs(ess_syz_dict[eta[6,9,9,3,3,3,4,4,4]],final_subs_dict,D)

# display(rel1)
# display(rel2)

branching_rel=collect((rel2-rel1*Rational(424216,860003*2))*Rational(4300015,624834),eta[6,9,9,4])
display(branching_rel)

In [ ]:
curr_syz={eta[3,9,6,4],eta[6,9,9,4,4],eta[6,9,9,3,3,4,4],eta[6,9,9,3,3,3,4,4,4]}.union(ess_syz_b2)
for a in curr_syz:
    display(ds_subs(ess_syz_dict[a],final_subs_dict,D))

In [ ]:
for a in ess_syz_b2:
    display(expand(eta[6,9,9,4]*ess_syz_dict[a]))

In [ ]:
ds_subs(ess_syz_dict[eta[6,9,9,3,3,4,4]],final_subs_dict,D)

In [ ]:
br_34=simplify(ds_subs(P.fund_der(P.fund_der(branching_rel,3),4),final_subs_dict,D))
br_34

In [ ]:
temp_dict={eta[3, 9, 6]: {(): 0}, eta[3, 9, 4]: {(): 0},
           eta[6,9,9]: {(3,3): 0,(4,4):0,(4,3): 6*eta[6,9,9,3,4],(3,4,3,4):Rational(2,15)*eta[6,9,9,4]*eta[6,9,9]}}

for a in ess_syz_b2:
    display(a)
    display(ds_subs(ess_syz_dict[a],final_subs_dict,D))
    display('--------------------------')

In [ ]:
factor(-3749004*eta[6, 9, 9, 3, 4]/4300015 + 624834*eta[6, 9, 9, 4, 3]/4300015)

### Zero-Wilczynski Case

In [35]:
zero_wilc_dict=copy.deepcopy(Bianchi_dict)
not_added=copy.copy(not_added_bookmark)

In [ ]:
ds_add_key(eta[3,9,6],0,zero_wilc_dict,D)
ds_add_key(eta[3,9,4],0,zero_wilc_dict,D)
print(ds_subs_needed(zero_wilc_dict))

In [ ]:
factor(not_added[0])

In [19]:
not_added_zero_Wilc_bookmark=copy.copy(not_added)

#### First Branch

In [20]:
b1_Bianchi_dict=copy.deepcopy(zero_wilc_dict)
not_added=not_added_zero_Wilc_bookmark
ds_add_key(eta[6,9,9,4],0,b1_Bianchi_dict,D)

In [ ]:
display(ds_subs(not_added[1],b1_Bianchi_dict,D))

In [26]:
ds_add_key(eta[6,9,9,3,4],0,b1_Bianchi_dict,D)

In [ ]:
display(ds_subs(not_added[2],b1_Bianchi_dict,D))

#### Second Branch

In [ ]:
b2_Bianchi_dict=copy.deepcopy(zero_wilc_dict)
not_added=not_added_zero_Wilc_bookmark
factor(not_added[0])

In [39]:
ds_add_key(eta[6,9,9,4,3],6*eta[6,9,9,3,4],b2_Bianchi_dict,D)

In [ ]:
display(ds_subs(not_added[3],b2_Bianchi_dict,D))

### Continuing

In [53]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Bianchi_dict'+'{j}'.format(j=7)]=Bianchi_dict

In [54]:
saved_Bianchi_dict=copy.deepcopy(Bianchi_dict) # Through wght 14 at the moment
saved_curv=copy.deepcopy(P.curvature)

In [55]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     temp=shelf['Bianchi_dict'+'{j}'.format(j=7)]

In [ ]:
for a in P.fund_invars:
    print(a, a in Bianchi_dict)

In [57]:
# Bianchi_dict=copy.deepcopy(saved_Bianchi_dict) # Through wght 13 at the moment
# P.curvature=copy.deepcopy(saved_curv)

In [58]:
P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
D.curv=-P.curvature[0]

In [ ]:
for w in range(1,10):
    print(w)
    check_Bianchi(w)

In [ ]:
# These should only be derivatives of the fundamental invariants
temp=set()
for k1 in Bianchi_dict:
    for k2 in Bianchi_dict[k1]:
        temp=temp.union(Indexed_obj_in_expr(Bianchi_dict[k1][k2]))
temp

In [63]:
zero_Wilc_dict={eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}}

In [ ]:
for a in Bianchi_dict[eta[6,9,9]]:
    print(a,'-->',Bianchi_dict[eta[6,9,9]][a])
    print()

In [ ]:
for a in syzygies_by_wght[9]:
    display('-------------------------')
    display(ds_subs(a,zero_Wilc_dict,D))
    display(ds_subs(ds_subs(ds_subs(a,zero_Wilc_dict,D),Bianchi_dict,D),zero_Wilc_dict,D))

In [12]:
I=IndexedBase('I')
W=IndexedBase('W')

fund_invar_dict={I[3]:eta[6,9,9],W[4]:eta[3,9,6],W[6]:eta[3,9,4]}

def convert_syzygy(syz):
    T=Indexed_obj_in_expr(syz)
    s={}
    for t in T:
        temp=fund_invar_dict[t.base[t.indices[0]]]
        s[t]=temp.base[temp.indices+t.indices[1:len(t.indices)]]
    return syz.xreplace(s)

In [ ]:
old_syzygies=[
    7*I[3, 3, 3] + W[4, 4],
    14*I[3, 3, 3, 3, 4] - 7*I[3, 3, 3, 4, 3] + W[4, 3, 4, 4],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -28*I[3, 3, 3, 3, 3, 3]/5 - 96*I[3, 3]*W[4]/5 - 12*I[3]*W[4, 3]/5 - 7*W[4, 3, 3, 3, 4]/30 + W[4, 3, 3, 4, 3] - 3*W[4, 3, 4, 3, 3]/2 + 49*W[6, 3, 4]/1650 - 14*W[6, 4, 3]/275
]

for old_syz in old_syzygies:
    print(simplify(ds_subs(convert_syzygy(old_syz),Bianchi_dict,D)))

In [ ]:
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'harmonic')
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'coexact')

In [ ]:
ds_subs(P.fund_der(eta[6,9,9],3),Bianchi_dict,D)

In [29]:
partial_Bianchi_dicts={20:copy.deepcopy(Bianchi_dict)}
for w in reversed(range(20)):
    r=copy.deepcopy(partial_Bianchi_dicts[w+1])
    for k in r:
        for j in list(r[k].keys()):
            if -wght_of_ind(k.base[k.indices+j],g)>w: r[k].pop(j)
    for k in list(r.keys()):
        if r[k]==dict(): r.pop(k)
    partial_Bianchi_dicts[w]=r

In [ ]:
harm_basis={}
for i in [3,4,6]:
    harm_basis[i]=C.elt({})
    for j in range(len(C.basis(2,i))):
        harm_basis[i]=harm_basis[i]+C.subspace_basis('harmonic',2,i)[j]*C.basis(2,i)[j]
